# HawkEye RAG — Day 1: The Naive Baseline

Goal (same as the guide): prove the concept — a chatbot that gets smarter by stuffing relevant text into the system prompt — before introducing any real infrastructure.

**Deviations from the guide, and why:**

1. **Real KB subset, not toy docs.** We're using the `knowledge-base/Hardware` folder from the real TeamDynamix scrape, filtered down to ~67 general-topic articles (room-specific classroom/conference instructions excluded — see the filtering work done just before this notebook).
2. **Frontmatter stripping.** Our `.md` files have YAML frontmatter (title, tags, url, etc.) that the guide's toy docs didn't have. We use `python-frontmatter` to cleanly split metadata from body text, so we don't embed raw YAML into the LLM's context.
3. **Multi-word keys → word → list-of-docs dict.** The guide's dict is `single word → one document` (works because employee surnames were unique). Our article titles are multi-word ("How-To-Use-A-SMART-Board"), so we split each filename slug into words, drop stopwords, and index `word → list of matching articles`. This is a tiny inverted index — a natural (and honest) evolution of the guide's approach once titles aren't single unique words.
4. **Gemini instead of OpenAI.** Same shape of call (system prompt + history + user message → model), different SDK. Gemini's chat call takes a `system_instruction` in its config rather than a system message in the messages list, and history has to be built as a list of `Content` objects with `role='user'`/`role='model'` (not `'assistant'`).

In [1]:
import sys
print(sys.executable)

d:\mrsha\Projects\HawkEye\.venv\Scripts\python.exe


In [2]:
import os
import re
from pathlib import Path

import frontmatter
import gradio as gr
from dotenv import load_dotenv
from google import genai
from google.genai import types

d:\mrsha\Projects\HawkEye\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Setting up

load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

MODEL = "gemini-2.5-flash-lite"  # cheap + stable, good fit for a naive baseline
client = genai.Client()

Google API Key exists and begins AQ.Ab8RN


### Load the filtered Hardware subset into a keyword-indexed dictionary

Same `room_pattern` filter we tested earlier, now doing the real work: strip frontmatter, extract keywords from the filename slug, index each article under every distinctive word in its title.

In [4]:
KB_PATH = Path("knowledge-base/Hardware")

room_pattern = re.compile(r'-Technology-Instructions-\d+\.md$')

STOPWORDS = {
    "a", "an", "the", "to", "of", "for", "and", "or", "on", "in", "your",
    "you", "is", "are", "how", "using", "use", "with", "at", "from", "by",
    "all", "other", "into", "up", "down", "new", "it",
}

def slug_keywords(filename: str) -> list[str]:
    stem = re.sub(r'-\d+$', '', Path(filename).stem)  # strip trailing article id
    words = re.split(r'[-\s]+', stem)
    words = [w.lower() for w in words if w]
    return [w for w in words if w not in STOPWORDS and len(w) > 1]

In [5]:
# knowledge: word -> list of (title, body) tuples
knowledge: dict[str, list[tuple[str, str]]] = {}

skipped = 0
loaded = 0

for filepath in sorted(KB_PATH.glob("*.md")):
    if room_pattern.search(filepath.name):
        skipped += 1
        continue
    post = frontmatter.load(filepath)
    title = post.get("title", filepath.stem)
    body = post.content
    loaded += 1
    for word in slug_keywords(filepath.name):
        knowledge.setdefault(word, []).append((title, body))

print(f"Loaded {loaded} articles, skipped {skipped} room-specific docs")
print(f"Indexed {len(knowledge)} distinct keywords")

Loaded 67 articles, skipped 189 room-specific docs
Indexed 196 distinct keywords


In [6]:
# Sanity check: what's under a couple of keywords?

print("'voicemail' ->", [title for title, _ in knowledge.get("voicemail", [])])
print("'print' ->", [title for title, _ in knowledge.get("print", [])])
print("'smart' ->", [title for title, _ in knowledge.get("smart", [])])

'voicemail' -> ['Voicemail Setup']
'print' -> ['Adding funds to your Print Quota', 'Managed Print Initiative - How to Enroll', 'Managed Print Initiative - How to Print (Using Pharos Secure Print)', 'Managed Print Initiative - How to Print (Using Pharos Secure Print)', 'Managed Print Initiative - How to Print (Using Pharos Secure Print)', 'Managed Print Initiative - How to Scan (Using Pharos Secure Print)', 'Managed Print Initiative - How to Scan (Using Pharos Secure Print)', 'Managed Print Initiative - Reset Pharos Passcode', 'Print Quota Refund', '* Wireless Printing (Print and pickup)', 'Wireless Printing: Web Print (Getting Started)']
'smart' -> ['How To Use A SMART Board', 'How To Use A Stand-Alone SMART Board']


### The naive retriever

Same idea as the guide's `get_relevant_context`: strip punctuation, lowercase, split the question into words, look each word up. The one change is we're now getting a *list* of matches per word (not a single doc), so we dedupe by title before returning.

In [7]:
def get_relevant_context(message: str) -> list[str]:
    text = ''.join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    matches: dict[str, str] = {}  # title -> body, dedupes automatically
    for word in words:
        for title, body in knowledge.get(word, []):
            matches[title] = body
    return list(matches.values())

In [8]:
get_relevant_context("How do I check my voicemail?")

['**To complete the initial setup of your voice mail system:**\n\n* Press the recording button\xa0![](http://support.newpaltz.edu/img/telecom/messages_button_edited_final.png)\n* When prompted for a pin, enter 12345#\n* To complete the setup, listen to the prompts from the phone and respond accordingly.\n* When you have completed all stages of the setup, you can hang up.']

In [9]:
get_relevant_context("How do I use the SMART Board and also fix my printer?")

['![Old Main SmartBoard](https://newpaltz.teamdynamix.com/TDPortal/Images/Viewer?fileName=11864c10-8b3d-4607-95ad-e094a8c4b183.jpg)\n\nSMART Board Operating\xa0Instructions\n\n* On the left side of the screen, click on the translucent arrows. Tools for using your SMART Board will become visible\n* Choose a pen\xa0to write on the screen\n* Choose the mouse option (arrow icon) and the pen (or your finger) will work as the\xa0computer mouse\n* The notepad option will open a blank page on which you may write or draw\n* If you wish, save your notations back to the computer or flash drive\n\nMany other functions of the SMART Board may be found by visiting their website at\xa0[www.smarttech.com](http://www.smarttech.com/ "smarttech.com").\n\nFor more information, walk in to the Service\xa0Desk in Humanities 103 or call 845-257-HELP (4357).  \nTo request\xa0technology assistance and support,\xa0**[submit a ticket](https://newpaltz.teamdynamix.com/TDClient/Requests/ServiceDet?ID=16187).**  \nTo

### Build the system prompt and wire up the chat function

In [10]:
SYSTEM_PREFIX = """
You are HawkEye, the IT help desk assistant for SUNY New Paltz.
You are an expert in answering questions about campus IT hardware, software, and services.
You are provided with additional context that might be relevant to the user's question.
Give brief, accurate answers. If you don't know the answer, say so — don't guess.

Relevant context:
"""

In [11]:
def additional_context(message: str) -> str:
    relevant_context = get_relevant_context(message)
    if not relevant_context:
        return "There is no additional context relevant to the user's question."
    result = "The following additional context might be relevant in answering the user's question:\n\n"
    result += "\n\n".join(relevant_context)
    return result

In [12]:
print(additional_context("How do I check my voicemail?"))

The following additional context might be relevant in answering the user's question:

**To complete the initial setup of your voice mail system:**

* Press the recording button ![](http://support.newpaltz.edu/img/telecom/messages_button_edited_final.png)
* When prompted for a pin, enter 12345#
* To complete the setup, listen to the prompts from the phone and respond accordingly.
* When you have completed all stages of the setup, you can hang up.


### The Gemini call

Two shape differences from the OpenAI-style call in the guide:
- The system prompt goes into `config=types.GenerateContentConfig(system_instruction=...)`, not into the messages list.
- Conversation history has to be built as `types.Content` objects, and Gemini calls the assistant's role `'model'` instead of `'assistant'` (Gradio's `gr.ChatInterface` gives us history in OpenAI's `{'role': 'assistant', ...}` shape, so we translate it).

In [13]:
def to_gemini_history(history: list[dict]) -> list[types.Content]:
    contents = []
    for m in history:
        role = "model" if m["role"] == "assistant" else "user"
        contents.append(types.Content(role=role, parts=[types.Part(text=m["content"])]))
    return contents

In [14]:
def chat(message: str, history: list[dict]) -> str:
    system_message = SYSTEM_PREFIX + additional_context(message)
    contents = to_gemini_history(history) + [types.Content(role="user", parts=[types.Part(text=message)])]
    response = client.models.generate_content(
        model=MODEL,
        contents=contents,
        config=types.GenerateContentConfig(system_instruction=system_message),
    )
    return response.text

In [15]:
chat("How do I check my voicemail?", [])

'To check your voicemail, dial 518-255-5000.'

## Now bring this up in Gradio's ChatInterface

Same as the guide — fastest way to get a working chat window, and Gradio manages the `history` list for us.

In [19]:
view = gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "d:\mrsha\Projects\HawkEye\.venv\Lib\site-packages\gradio\queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\mrsha\Projects\HawkEye\.venv\Lib\site-packages\gradio\route_utils.py", line 393, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\mrsha\Projects\HawkEye\.venv\Lib\site-packages\gradio\blocks.py", line 2280, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\mrsha\Projects\HawkEye\.venv\Lib\site-packages\gradio\blocks.py", line 1655, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\mrsha\Projects\HawkEye\.venv\Lib\site-packages\gradio\utils.py", line 1043, in async_wrapper
    response = await f(*args, **kwargs)
               ^^^^^^^^^^^^^

## Limitation to notice (same as the guide's Day 1 takeaway)

This only works if the question contains a word that's an indexed keyword. Try asking about "printer" (singular) vs. our indexed "print"/"printing" — or ask a question that spans two topics, or uses a synonym the KB doesn't. That gap is exactly what motivates Day 2's real chunking + embeddings.